In [ ]:
"""
=============================================================================
  Plant Disease Classification - Deep Learning with TensorFlow/Keras
  Models: ResNet50, VGG16, EfficientNetB0, MobileNetV2, Custom CNN
  Dataset: New Plant Diseases Dataset (Augmented) - 38 classes
=============================================================================
"""

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, VGG16, EfficientNetB0, MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  CONFIGURATION  — Edit this section to match your setup
# ─────────────────────────────────────────────────────────────
CONFIG = {
    # ── Paths ──────────────────────────────────────────────────
    "ROOT_DIR": r"D:\Leaf dataset 2\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)",
    "OUTPUT_DIR": r"D:\Leaf dataset 2\outputs",

    # ── Training ───────────────────────────────────────────────
    "IMG_SIZE": 224,          # All models use 224×224
    "BATCH_SIZE": 32,
    "EPOCHS": 30,
    "NUM_CLASSES": 38,
    "LEARNING_RATE": 1e-3,
    "SEED": 42,

    # ── Models to train (set False to skip any) ────────────────
    "MODELS_TO_RUN": {
        "CustomCNN":      True,
        "MobileNetV2":    True,
        "EfficientNetB0": True,
        "ResNet50":       True,
        "VGG16":          True,
    }
}

# ─────────────────────────────────────────────────────────────
#  REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────
tf.random.set_seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])

TRAIN_DIR = os.path.join(CONFIG["ROOT_DIR"], "train")
VALID_DIR = os.path.join(CONFIG["ROOT_DIR"], "valid")
TEST_DIR  = os.path.join(CONFIG["ROOT_DIR"], "test")
OUT_DIR   = CONFIG["OUTPUT_DIR"]
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE   = CONFIG["IMG_SIZE"]
BATCH_SIZE = CONFIG["BATCH_SIZE"]
EPOCHS     = CONFIG["EPOCHS"]
NUM_CLASSES= CONFIG["NUM_CLASSES"]
LR         = CONFIG["LEARNING_RATE"]

# ─────────────────────────────────────────────────────────────
#  DATA LOADERS
# ─────────────────────────────────────────────────────────────
def build_datasets():
    """Load train / validation / test datasets with augmentation on train set."""

    # Training — with augmentation
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=True,
        seed=CONFIG["SEED"],
    )

    valid_ds = tf.keras.utils.image_dataset_from_directory(
        VALID_DIR,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=False,
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=False,
    )

    class_names = train_ds.class_names
    print(f"\n✅  Found {len(class_names)} classes.")

    # Augmentation layer (applied only to train set)
    augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.15),
        layers.RandomBrightness(0.1),
        layers.RandomContrast(0.1),
    ], name="augmentation")

    # Normalise to [0, 1]
    normalize = layers.Rescaling(1.0 / 255)

    def preprocess_train(x, y):
        x = augmentation(x, training=True)
        x = normalize(x)
        return x, y

    def preprocess_eval(x, y):
        x = normalize(x)
        return x, y

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    valid_ds = valid_ds.map(preprocess_eval,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    test_ds  = test_ds.map(preprocess_eval,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

    return train_ds, valid_ds, test_ds, class_names


# ─────────────────────────────────────────────────────────────
#  MODEL BUILDERS  (trained from scratch)
# ─────────────────────────────────────────────────────────────

def build_custom_cnn(num_classes):
    """A compact custom CNN built from scratch."""
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, 3, padding="same", activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Block 4
        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),

        # Classifier
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ], name="CustomCNN")
    return model


def build_mobilenetv2(num_classes):
    """MobileNetV2 architecture trained from scratch (no pretrained weights)."""
    base = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights=None,          # from scratch
        classes=num_classes,
    )
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="MobileNetV2_scratch")


def build_efficientnetb0(num_classes):
    """EfficientNetB0 trained from scratch."""
    base = EfficientNetB0(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights=None,
        classes=num_classes,
    )
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="EfficientNetB0_scratch")


def build_resnet50(num_classes):
    """ResNet50 trained from scratch."""
    base = ResNet50(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights=None,
        classes=num_classes,
    )
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="ResNet50_scratch")


def build_vgg16(num_classes):
    """VGG16 trained from scratch."""
    base = VGG16(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights=None,
        classes=num_classes,
    )
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="VGG16_scratch")


MODEL_REGISTRY = {
    "CustomCNN":      build_custom_cnn,
    "MobileNetV2":    build_mobilenetv2,
    "EfficientNetB0": build_efficientnetb0,
    "ResNet50":       build_resnet50,
    "VGG16":          build_vgg16,
}


# ─────────────────────────────────────────────────────────────
#  CALLBACKS
# ─────────────────────────────────────────────────────────────
def get_callbacks(model_name, save_dir):
    ckpt_path = os.path.join(save_dir, f"{model_name}_best.keras")
    cb = [
        callbacks.ModelCheckpoint(
            filepath=ckpt_path,
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
        ),
        callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=7,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),
        callbacks.CSVLogger(
            os.path.join(save_dir, f"{model_name}_log.csv")
        ),
    ]
    return cb


# ─────────────────────────────────────────────────────────────
#  PLOTTING HELPERS
# ─────────────────────────────────────────────────────────────
def plot_history(history, model_name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{model_name} — Training History", fontsize=14)

    # Accuracy
    axes[0].plot(history.history["accuracy"],     label="Train Accuracy")
    axes[0].plot(history.history["val_accuracy"], label="Val Accuracy")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    # Loss
    axes[1].plot(history.history["loss"],     label="Train Loss")
    axes[1].plot(history.history["val_loss"], label="Val Loss")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{model_name}_history.png"), dpi=150)
    plt.close()


def plot_confusion_matrix(cm, class_names, model_name, save_dir):
    fig, ax = plt.subplots(figsize=(20, 18))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, linewidths=0.3,
    )
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True",      fontsize=12)
    ax.set_title(f"{model_name} — Confusion Matrix", fontsize=14)
    plt.xticks(rotation=90, fontsize=7)
    plt.yticks(rotation=0,  fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{model_name}_confusion_matrix.png"), dpi=150)
    plt.close()


# ─────────────────────────────────────────────────────────────
#  EVALUATION
# ─────────────────────────────────────────────────────────────
def evaluate_model(model, test_ds, class_names, model_name, save_dir):
    print(f"\n📊  Evaluating {model_name} on test set …")
    y_true, y_pred = [], []
    for x_batch, y_batch in test_ds:
        preds = model.predict(x_batch, verbose=0)
        y_pred.extend(np.argmax(preds, axis=1))
        y_true.extend(np.argmax(y_batch.numpy(), axis=1))

    # Classification report
    report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    report_path = os.path.join(save_dir, f"{model_name}_classification_report.txt")
    with open(report_path, "w") as f:
        f.write(report)
    print(report)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plot_confusion_matrix(cm, class_names, model_name, save_dir)

    # Overall accuracy
    accuracy = np.sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
    return accuracy


# ─────────────────────────────────────────────────────────────
#  COMPARISON CHART  (generated after all models finish)
# ─────────────────────────────────────────────────────────────
def plot_comparison(results: dict, save_dir: str):
    if not results:
        return
    names      = list(results.keys())
    val_accs   = [results[n]["val_accuracy"]  for n in names]
    test_accs  = [results[n]["test_accuracy"] for n in names]
    train_time = [results[n]["train_time_min"] for n in names]

    x = np.arange(len(names))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Model Comparison — Plant Disease Classification", fontsize=14)

    # Accuracy bars
    axes[0].bar(x - width/2, val_accs,  width, label="Val Accuracy",  color="#4C72B0")
    axes[0].bar(x + width/2, test_accs, width, label="Test Accuracy", color="#55A868")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(names, rotation=15)
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1)
    axes[0].legend()
    axes[0].set_title("Accuracy Comparison")
    for i, (v, t) in enumerate(zip(val_accs, test_accs)):
        axes[0].text(i - width/2, v + 0.005, f"{v:.3f}", ha="center", fontsize=8)
        axes[0].text(i + width/2, t + 0.005, f"{t:.3f}", ha="center", fontsize=8)

    # Training time
    axes[1].bar(names, train_time, color="#C44E52")
    axes[1].set_ylabel("Training Time (minutes)")
    axes[1].set_title("Training Time Comparison")
    axes[1].set_xticklabels(names, rotation=15)
    for i, t in enumerate(train_time):
        axes[1].text(i, t + 0.2, f"{t:.1f} min", ha="center", fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "model_comparison.png"), dpi=150)
    plt.close()
    print(f"\n📊  Comparison chart saved → {save_dir}/model_comparison.png")


# ─────────────────────────────────────────────────────────────
#  MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  Plant Disease Classification — TensorFlow/Keras")
    print("=" * 60)
    print(f"  GPU available: {tf.config.list_physical_devices('GPU')}")

    # Load data once — shared across all models
    print("\n📂  Loading datasets …")
    train_ds, valid_ds, test_ds, class_names = build_datasets()

    all_results = {}

    for model_name, builder_fn in MODEL_REGISTRY.items():
        if not CONFIG["MODELS_TO_RUN"].get(model_name, False):
            print(f"\n⏭  Skipping {model_name} (disabled in config)")
            continue

        print(f"\n{'='*60}")
        print(f"  Training: {model_name}")
        print(f"{'='*60}")

        save_dir = os.path.join(OUT_DIR, model_name)
        os.makedirs(save_dir, exist_ok=True)

        # Build model
        model = builder_fn(NUM_CLASSES)
        model.compile(
            optimizer=optimizers.Adam(learning_rate=LR),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
        )
        model.summary()

        # Train
        t0 = time.time()
        history = model.fit(
            train_ds,
            validation_data=valid_ds,
            epochs=EPOCHS,
            callbacks=get_callbacks(model_name, save_dir),
        )
        elapsed_min = (time.time() - t0) / 60

        # Plot training history
        plot_history(history, model_name, save_dir)

        # Best val accuracy from history
        best_val_acc = max(history.history["val_accuracy"])

        # Evaluate on test set
        test_accuracy = evaluate_model(model, test_ds, class_names, model_name, save_dir)

        # Save model
        model.save(os.path.join(save_dir, f"{model_name}_final.keras"))

        # Collect results
        result = {
            "val_accuracy":   round(best_val_acc,  4),
            "test_accuracy":  round(test_accuracy, 4),
            "train_time_min": round(elapsed_min,   2),
            "epochs_run":     len(history.history["loss"]),
        }
        all_results[model_name] = result
        print(f"\n✅  {model_name} done | Val Acc: {best_val_acc:.4f} | "
              f"Test Acc: {test_accuracy:.4f} | Time: {elapsed_min:.1f} min")

        # Save individual result
        with open(os.path.join(save_dir, "result.json"), "w") as f:
            json.dump(result, f, indent=2)

    # ── Summary ─────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("  FINAL RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"{'Model':<20} {'Val Acc':>10} {'Test Acc':>10} {'Time (min)':>12} {'Epochs':>8}")
    print("-" * 65)
    for name, res in all_results.items():
        print(f"{name:<20} {res['val_accuracy']:>10.4f} {res['test_accuracy']:>10.4f} "
              f"{res['train_time_min']:>12.1f} {res['epochs_run']:>8}")

    # Save summary JSON
    summary_path = os.path.join(OUT_DIR, "all_results_summary.json")
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\n💾  Summary saved → {summary_path}")

    # Comparison chart
    plot_comparison(all_results, OUT_DIR)
    print("\n🎉  All models completed successfully!")


if __name__ == "__main__":
    main()